# Person 5 – Product Feature Engineering
This notebook extracts structured product attributes, price bands, room groups, category hierarchy, and text suitable for future embeddings.

**Input:** `data/raw/products.csv` (preferred to preserve readable categories) or `data/processed/products_clean.csv` as fallback.
**Outputs:** `outputs/product_features.csv` and `outputs/product_features.parquet`.

Note: dimensions are retained when present. Missing dimensions are not fabricated.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.person5_product_feature_engineering import load_products, engineer_product_features, OUTPUT_DIR


## 1. Load the product catalogue


In [2]:
products = load_products()
print('Dataset shape:', products.shape)
print('Columns:', products.columns.tolist())
display(products.head())


Reading: C:\Users\hkush\nish\GenAI_Recommender_-FrameWorksA-\data\raw\products.csv
Dataset shape: (21, 10)
Columns: ['product_id', 'product_name', 'category', 'material', 'colour', 'style', 'price', 'discount', 'rating', 'room_type']


,product_id,product_name,category,material,colour,style,price,discount,rating,room_type
0,P001,Modern Sofa,Sofa,Fabric,Beige,Modern,28500.0,10.0,4.5,Living Room
1,P002,Wooden Dining Table,Dining Table,Sheesham Wood,Brown,Traditional,32000.0,NaN,4.6,Dining Room
2,P003,Queen Size Bed,Bed,Engineered Wood,Walnut,Contemporary,41000.0,5.0,4.4,Bedroom
3,P004,Office Chair,Chair,Mesh,Black,Modern,NaN,10.0,4.2,Study Room
4,P005,Bookshelf,Storage,Plywood,White,Minimalist,8500.0,0.0,4.1,Study Room


## 2. Inspect available product attributes


In [3]:
display(products.isnull().sum().to_frame('missing_values'))
for col in ['category', 'material', 'colour', 'style', 'room_type']:
    if col in products.columns:
        print(f'\\n{col}:')
        print(products[col].value_counts(dropna=False).head(15))


,missing_values
product_id,0
product_name,0
category,0
material,0
colour,0
style,0
price,1
discount,1
rating,0
room_type,0


\ncategory:
category
Storage         6
Chair           4
Table           3
Sofa            2
Bed             2
Dining Table    1
sofa            1
Lighting        1
Bench           1
Name: count, dtype: int64
\nmaterial:
material
Plywood            4
Engineered Wood    3
Solid Wood         3
Fabric             2
Teak Wood          2
Sheesham Wood      1
Mesh               1
Velvet             1
MDF                1
Leather            1
Metal              1
Linen              1
Name: count, dtype: int64
\ncolour:
colour
Brown         4
White         4
Beige         2
Black         2
Natural       2
Walnut        1
Green         1
Grey          1
Oak           1
Dark Brown    1
Gold          1
Blue          1
Name: count, dtype: int64
\nstyle:
style
Modern          7
Traditional     4
Contemporary    3
Minimalist      3
Rustic          2
Industrial      1
Scandinavian    1
Name: count, dtype: int64
\nroom_type:
room_type
Living Room    8
Bedroom        4
Study Room     4
Dining Room    2

## 3. Engineer product features
- Extract dimensions when available.
- Create Budget, Mid-Range, and Premium price categories.
- Create category hierarchy and broad room classification.
- Generate descriptions and `embedding_text` for future semantic search.


In [4]:
product_features = engineer_product_features(products)
display(product_features.head())
print('Feature dataset shape:', product_features.shape)


,product_id,product_name,category,category_group,category_subgroup,material,colour,style,room_type,room_group,price,price_category,discount,rating,length_cm,width_cm,height_cm,dimensions_cm,product_description,embedding_text
0,P001,Modern Sofa,Sofa,Living Room Furniture,Seating,Fabric,Beige,Modern,Living Room,Living Area,28500.0,Mid-Range,10.0,4.5,<NA>,<NA>,<NA>,Not available,Modern Sofa is a Modern sofa designed for the ...,Product: Modern Sofa. Category: Sofa. Category...
1,P002,Wooden Dining Table,Dining Table,Dining Room Furniture,Dining,Sheesham Wood,Brown,Traditional,Dining Room,Dining Area,32000.0,Mid-Range,NaN,4.6,<NA>,<NA>,<NA>,Not available,Wooden Dining Table is a Traditional dining ta...,Product: Wooden Dining Table. Category: Dining...
2,P003,Queen Size Bed,Bed,Bedroom Furniture,Sleeping,Engineered Wood,Walnut,Contemporary,Bedroom,Bedroom,41000.0,Premium,5.0,4.4,<NA>,<NA>,<NA>,Not available,Queen Size Bed is a Contemporary bed designed ...,Product: Queen Size Bed. Category: Bed. Catego...
3,P004,Office Chair,Chair,Seating Furniture,Chair,Mesh,Black,Modern,Study Room,Work Area,NaN,Unknown,10.0,4.2,<NA>,<NA>,<NA>,Not available,Office Chair is a Modern chair designed for th...,Product: Office Chair. Category: Chair. Catego...
4,P005,Bookshelf,Storage,Other Furniture,Other,Plywood,White,Minimalist,Study Room,Work Area,8500.0,Budget,0.0,4.1,<NA>,<NA>,<NA>,Not available,Bookshelf is a Minimalist storage designed for...,Product: Bookshelf. Category: Storage. Categor...


Feature dataset shape: (20, 20)


## 4. Validate the engineered dataset


In [5]:
print('Duplicate product IDs:', product_features['product_id'].duplicated().sum())
print('Price categories:')
display(product_features['price_category'].value_counts(dropna=False))
print('Room groups:')
display(product_features['room_group'].value_counts(dropna=False))
display(product_features[['product_id','product_name','category','material','style','room_type','price_category','dimensions_cm']].head(10))


Duplicate product IDs: 0
Price categories:


price_category
Mid-Range    10
Budget        7
Premium       2
Unknown       1
Name: count, dtype: int64

Room groups:


room_group
Living Area                7
Bedroom                    4
Work Area                  4
Dining Area                2
Entryway                   2
Multi-purpose / Unknown    1
Name: count, dtype: int64

,product_id,product_name,category,material,style,room_type,price_category,dimensions_cm
0,P001,Modern Sofa,Sofa,Fabric,Modern,Living Room,Mid-Range,Not available
1,P002,Wooden Dining Table,Dining Table,Sheesham Wood,Traditional,Dining Room,Mid-Range,Not available
2,P003,Queen Size Bed,Bed,Engineered Wood,Contemporary,Bedroom,Premium,Not available
3,P004,Office Chair,Chair,Mesh,Modern,Study Room,Unknown,Not available
4,P005,Bookshelf,Storage,Plywood,Minimalist,Study Room,Budget,Not available
5,P006,Coffee Table,Table,Solid Wood,Rustic,Living Room,Budget,Not available
6,P007,Two Seater Sofa,sofa,Velvet,Modern,Living Room,Mid-Range,Not available
7,P008,Bedside Table,Storage,MDF,Minimalist,Bedroom,Budget,Not available
8,P009,Recliner Chair,Chair,Leather,Contemporary,Living Room,Mid-Range,Not available
9,P010,Wardrobe,Storage,Engineered Wood,Modern,Bedroom,Mid-Range,Not available


## 5. Save final deliverables


In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
product_features.to_csv(OUTPUT_DIR / 'product_features.csv', index=False)
try:
    product_features.to_parquet(OUTPUT_DIR / 'product_features.parquet', index=False)
    print('Saved CSV and Parquet.')
except ImportError:
    print('CSV saved. Install pyarrow to write Parquet: pip install pyarrow')


Saved CSV and Parquet.
